In [ ]:
import pandas as pd
decam_objects = pd.read_csv("../../data/decam_objects.csv")
decam_objects

In [ ]:
import matplotlib.pyplot as plt                                                                                                                                                
                                                                                                                                                                                
fig, axes = plt.subplots(1, 2, figsize=(14, 4))                                                                                                                                
                                                                                                                                                                            
axes[0].hist(decam_objects["ra"].dropna(), bins=100)
axes[0].set_xlabel("RA (deg)")
axes[0].set_ylabel("Count")
axes[0].set_title("RA distribution")

axes[1].hist(decam_objects["dec"].dropna(), bins=100)                                                                                                                          
axes[1].set_xlabel("Dec (deg)")
axes[1].set_title("Dec distribution")                                                                                                                                          
                                                                                                                                                                            
plt.tight_layout()
plt.show()

In [ ]:
import importlib
# Reload the module to reflect changes
import thor.utils.rubin_stats_visualizations
importlib.reload(thor.utils.rubin_stats_visualizations)

In [ ]:
# plot on skymap
from thor.utils.rubin_stats_visualizations import plot_skymap

plot_skymap(                                                                                                                                                                   
    ra=decam_objects["ra"].values,                                                                                                                                             
    dec=decam_objects["dec"].values,                                                                                                                                           
    title="DECam objects + catalog footprints",                                                                                                                                
    heatmap=True,                                                                                                                                                              
    bin_size_deg=1.0,                                                                                                                                                          
    plot_catalogs=True,                                                                                                                                                        
    plot_ddf=True,                                                                                                                                                             
)   

In [ ]:
# open matches returned by generic_crossmatch.py
# python -m thor.generic_crossmatch data/decam_objects.csv --method conesearch --save

import json                                                                                                                                                                    
                                                                                                                                                                                
with open("../../data/crossmatch_results/decam_objects_crossmatch_20261021T151805Z.json") as f:                                                                                
    crossmatch_results = json.load(f)

In [ ]:
# format as DF

import pandas as pd

Z_COLS     = ["z", "Z_BEST", "ZPHOT", "zfinal", "zpdf_med"]                                                                                                                    
Z_UNC_COLS = ["z_unc", "z_err", "z_phot_err", "zphot_err", "ez_best", "redshift_err"]
I_MAG_COLS = ["i", "MAG_AUTO_i", "mag_i", "i_mag"]                                                                                                                             
I_ERR_COLS = ["i_err", "MAGERR_AUTO_i", "magerr_i", "i_mag_err"]                                                                                                             

def _get_first(d, keys):
    for k in keys:
        if k in d and d[k] is not None:
            return d[k]
    return None

rows = []
for objid, catalogs in crossmatch_results.items():
    matches = {cat: data for cat, data in catalogs.items() if data is not None}
    if not matches:
        continue

    cat_names = list(matches.keys())
    catalog = cat_names if len(cat_names) > 1 else cat_names[0]

    # use closest match for per-object columns
    best = min(matches.values(), key=lambda d: d.get("conesearch_arcsecs", float("inf")))

    rows.append({
        "objid":              objid,
        "catalog":            catalog,
        "id_cat":              best.get("id"),
        "ra_cat":             best.get("ra"),                                                                                                                                  
        "dec_cat":            best.get("dec"),                                                                                                                                 
        "z_cat":              _get_first(best, Z_COLS),                                                                                                                        
        "z_unc_cat":          _get_first(best, Z_UNC_COLS),                                                                                                                    
        "conesearch_arcsecs": best.get("conesearch_arcsecs"),                                                                                                                  
        "i_mag":              _get_first(best, I_MAG_COLS),                                                                                                                    
        "i_err_mag":          _get_first(best, I_ERR_COLS),                                                                                                                    
    })                                                                                                                                                                         
                                                                                                                                                                                
df_crossmatches = pd.DataFrame(rows)                                                                                                                                           
print(f"{len(df_crossmatches):,} objects with at least one catalog crossmatch")
df_crossmatches_tight = df_crossmatches[df_crossmatches["conesearch_arcsecs"] <= 2.0]     
print(f"{len(df_crossmatches_tight):,} objects with at least one catalog crossmatch within 2 arcsec")
df_crossmatches_tight.head()

In [ ]:
# save just the objids

# import json                                                                                                                                                                    
                                                                                                                                                                                                                                                                                                                                  
# objids = list(crossmatch_results.keys())                                                                                                                                       
                                                                                                                                                                                
# with open("../../data/decam_crossmatches.json", "w") as f:                                                                                                                     
#     json.dump(objids, f, indent=2) 


# save the df
# df_crossmatches_tight.to_csv("../../data/df_decam_crossmatches.csv", index=False)